# MobileViTv2-0.75x 推理与评估（10 类，激活函数 ReLU）

本文件对应训练脚本 `train.ipynb` 的产出：宽度 **0.75x**、激活 **relu**、**10 类**（c0~c9）。

**按要求做的调整**

1. `WIDTH_MULTIPLIER = 0.75`（原来是 1.0，与预训练/微调权重不符，会直接加载失败）；
2. 激活函数 `relu`，且与训练一致地同时写 `model.activation.*` 与 `model.classification.activation.*`；
3. 类别数不再写死 1000，改为从数据集读取（10），并与检查点自带的 `config` 交叉校验；
4. 测试集固定为 `C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p`。

**修复的原文件问题**

| 问题 | 说明 |
| --- | --- |
| 单元必然报错 `NameError: transform_list` | 原"单图/单模型评估"单元先 `transforms.Compose` 建好 `transform`，紧接着又对**从未定义**的 `transform_list` 做 `+=`，该单元任何情况下都跑不完 |
| 预处理与训练不一致 | 该单元用 `Resize(288) + CenterCrop(256)`，而训练只有 `ToTensor + Normalize`；现在两边严格一致（数据本身是 256×256，`RESIZE=None`） |
| 分类数写死 1000 | `num_classes = 1000` 与 10 类数据集不符，评估函数里的逐类统计会错位 |
| 静默加载部分权重 | 原代码在加载失败时退回"只加载形状匹配的层"，于是一个 1000 类旧权重会被当成 10 类模型评估，准确率其实是随机分类头的结果；现在分类头加载不上**直接报错** |
| 类别顺序不校验 | ImageFolder 按目录名排序，训练/测试类别顺序不一致时准确率会整体错位；现在与检查点里的 `class_names` 逐项比对 |
| 1000 类混淆矩阵 | `figsize=(max(8, num_classes), ...)` 在 1000 类时会画出巨大的图；现在按类别数自适应并关闭 figure |
| 硬编码 `pre.pt` / 1000 类 ONNX | ONNX 导出与评估单元同步改为 10 类 / 0.75x / relu，并显式使用 TorchScript 导出器（见下） |
| 其它 | `exit(1)` 会直接结束内核、`torch.cuda.amp.autocast` 在新版 torch 中的弃用、无匹配文件时不给出可用回退、结果字典缺少预测值 |

**运行顺序**：先运行第 1 个代码单元（公共函数），再按需运行后面的评估 / ONNX 单元。

**关于 ONNX 导出**：本机 torch 2.9.1 的 `torch.onnx.export` 默认 `dynamo=True`，实测有两个坑：
导出器会打印 emoji，在 GBK 控制台下直接抛 `UnicodeEncodeError`；而且它不接受 `dynamic_axes`。
因此导出单元里显式使用 `dynamo=False`（TorchScript 导出器），实测导出后可动态 batch、与 PyTorch 输出最大误差 1.3e-06。
注意该导出器已被官方标记为"将来会移除"，以后升级 torch 时记得改用 `dynamic_shapes`。

---

## 本次修订（配合训练脚本的检查点改动）

训练脚本现在把两份权重分开保存：`model_state_dict` 是原始训练权重（续训用），`ema_model_state_dict` 是 EMA 权重
（推理/部署优先用），并写入 `inference_weights` 字段说明推荐哪一份。因此本文件的 `extract_state_dict()` 也做了对应调整：

- 优先按 `inference_weights` 指定的那份取权重；没有该字段时**默认优先 EMA**（EMA 通常比原始权重更稳、更准，
  评估与导出用的就是它）；取用时会打印"从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）"。
- 纯权重文件（例如用"提取纯权重"单元导出的 `.pth`）本身没有包装，照旧直接使用。
- 这样无论检查点是新格式还是你以前存的旧格式（旧格式里 `model_state_dict` 存的就是 EMA 权重），
  推理端选到的都是同一份"更准"的权重。


In [2]:
# ==================== 公共配置与函数（先运行本单元） ====================
import glob
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

try:
    from cvnets.models.classification.mobilevit_v2 import MobileViTv2
except ImportError as exc:
    raise ImportError(
        "当前内核找不到 cvnets，请把本笔记本的内核切到 conda 的 py312 环境，"
        "并在该环境中安装 ml-cvnets（例如 pip install -e C:/transformerFPGA/MobileViT/ml-cvnets）。"
    ) from exc

warnings.filterwarnings('ignore')

# ---- 与训练脚本保持一致的结构与预处理 ----
TEST_DATA_DIR = r'C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p'
WIDTH_MULTIPLIER = 0.75
ACTIVATION = 'relu'
RESIZE = None                 # 训练时没有 Resize（数据本身 256x256），推理必须一致
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

BATCH_SIZE = 16
NUM_WORKERS = 0
USE_AMP = False               # relu 模型可开 bf16；先用 False 核对正确性
RESULT_DIR = 'model_evaluation_results'


def build_opts(num_classes: int, width_multiplier: float = WIDTH_MULTIPLIER,
               act_name: str = ACTIVATION):
    """与训练脚本完全相同的模型构建选项（宽度 0.75x、激活 relu）。

    注意：直接调用 ``MobileViTv2(opts)`` 时真正生效的是 ``model.activation.name``；
    ``model.classification.activation.name`` 只在 ``build_model()`` 内部被复制过去，两处都写。
    激活函数（ReLU/Swish）不含参数，所以换成 relu 不会改变任何 state_dict 键。
    """
    import argparse
    opts = argparse.Namespace()
    attrs = {
        "model.classification.n_classes": num_classes,
        "model.classification.gradient_checkpointing": False,
        "model.classification.enable_layer_wise_lr_decay": False,
        "model.classification.layer_wise_lr_decay_rate": 1.0,
        "model.classification.freeze_batch_norm": False,
        "model.classification.pretrained": None,
        "model.classification.finetune_pretrained_model": False,
        "model.classification.n_pretrained_classes": None,
        "model.classification.classifier_dropout": 0.1,
        "model.classification.activation.name": act_name,
        "model.classification.activation.inplace": False,
        "model.classification.activation.neg_slope": 0.1,
        "model.activation.name": act_name,
        "model.activation.inplace": False,
        "model.activation.neg_slope": 0.1,
        "model.normalization.name": "batch_norm",
        "model.normalization.momentum": 0.1,
        "model.normalization.eps": 1e-5,
        "model.normalization.groups": 32,
        "model.layer.conv_init": "kaiming_normal",
        "model.layer.linear_init": "trunc_normal",
        "model.layer.pretrained": None,
        "model.layer.global_pool": "mean",
        "model.classification.mitv2.width_multiplier": width_multiplier,
        "model.classification.mitv2.attn_dropout": 0.1,
        "model.classification.mitv2.ffn_dropout": 0.1,
        "model.classification.mitv2.dropout": 0.2,
        "model.classification.mitv2.attn_norm_layer": "layer_norm_2d",
        "model.layer.deit": None,
        "common.enable_coreml_compatible_module": False,
        "model.segmentation.n_classes": None,
        "common.regnet.quantize": False,
        "common.mixed_precision": False,
    }
    for k, v in attrs.items():
        setattr(opts, k, v)
    return opts


def build_transform(resize=RESIZE):
    """必须与训练时一致：Resize(可选) -> ToTensor -> ImageNet 归一化。"""
    ops = []
    if resize is not None:
        ops.append(transforms.Resize(resize))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
    ]
    return transforms.Compose(ops)


def extract_state_dict(ckpt):
    """从"纯权重文件"或"完整检查点"中取出权重，并去掉 ``module.`` 前缀。

    完整检查点里有两份权重，用途不同：
    * ``model_state_dict``     —— 原始训练权重（续训用）
    * ``ema_model_state_dict`` —— EMA 权重（**推理通常更好，优先用它**）

    检查点里的 ``inference_weights`` 字段指出训练脚本推荐哪一份；没有该字段时默认优先 EMA。
    """
    if not isinstance(ckpt, dict):
        raise TypeError(f"无法识别的权重文件类型: {type(ckpt)}")

    if all(torch.is_tensor(v) for v in ckpt.values()):
        state = ckpt  # 纯权重文件，本身就是 state_dict
    else:
        preferred = ckpt.get('inference_weights', 'ema')
        if preferred == 'model':
            keys = ('model_state_dict', 'ema_model_state_dict')
        else:
            keys = ('ema_model_state_dict', 'model_state_dict')
        state = None
        for key in keys + ('state_dict', 'model', 'module'):
            inner = ckpt.get(key)
            if isinstance(inner, dict) and all(torch.is_tensor(v) for v in inner.values()):
                state = inner
                print(f"  从键 '{key}' 中提取模型权重"
                      + ("（EMA 权重，推理用）" if key == 'ema_model_state_dict' else ""))
                break
        if state is None:
            raise ValueError(f"无法在权重文件中找到 state_dict，可用键: {list(ckpt.keys())}")

    return {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}


def load_checkpoint_into_model(model, checkpoint_path, allow_missing_head=False):
    """把检查点/纯权重加载进模型。

    分类头形状不匹配（例如误把 1000 类的 ImageNet 权重当成本次微调结果）时**直接报错**，
    而不是静默地只加载 backbone —— 否则评估出来的准确率是随机分类头的结果，毫无意义。
    """
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"找不到模型文件: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {}) if isinstance(ckpt, dict) else {}
    state = extract_state_dict(ckpt)

    model_state = model.state_dict()
    loadable, skipped = {}, {}
    for k, v in state.items():
        if k in model_state and tuple(model_state[k].shape) == tuple(v.shape):
            loadable[k] = v
        elif k in model_state:
            skipped[k] = f"形状不匹配 {tuple(v.shape)} -> {tuple(model_state[k].shape)}"
        else:
            skipped[k] = "模型中不存在该键"

    skipped_head = [k for k in skipped if k.startswith('classifier.')]
    skipped_other = [k for k in skipped if not k.startswith('classifier.')]
    if skipped_other:
        raise RuntimeError(
            "权重与模型结构不匹配（除分类头外还有键对不上），"
            f"请检查 width_multiplier / attn_norm_layer 等配置：{skipped_other[:10]}"
        )
    if skipped_head and not allow_missing_head:
        head_shape = tuple(state['classifier.1.weight'].shape) if 'classifier.1.weight' in state else '未知'
        raise RuntimeError(
            f"分类头没有加载上：{skipped_head}（权重里是 {head_shape}，"
            f"当前模型是 {model.classifier[-1].out_features} 类）。"
            "这通常说明加载的不是本次微调产出的检查点；继续评估只会得到随机分类头的结果。"
            "确认无误时可显式传入 allow_missing_head=True。"
        )

    model_state.update(loadable)
    model.load_state_dict(model_state, strict=True)
    print(f"  已加载 {len(loadable)}/{len(state)} 个张量" +
          (f"，跳过分类头 {skipped_head}" if skipped_head else ""))
    return cfg


def build_model_from_checkpoint(checkpoint_path, num_classes, device=None):
    """按检查点自带的 config 构建模型并加载权重，返回 (model, cfg)。"""
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    preview = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = preview.get('config', {}) if isinstance(preview, dict) else {}

    width = cfg.get('width_multiplier', WIDTH_MULTIPLIER)
    act = cfg.get('activation', ACTIVATION)
    n_cls = cfg.get('num_classes', num_classes)

    if cfg:
        print(f"检查点配置: 类别数={n_cls} 宽度={width} 激活={act} 类别名={cfg.get('class_names')}")
        if n_cls != num_classes:
            raise RuntimeError(
                f"检查点是 {n_cls} 类模型，而数据集有 {num_classes} 类，类别索引无法对齐，已终止。"
            )
        if act != ACTIVATION:
            print(f"警告: 检查点激活为 {act}，与当前设置 {ACTIVATION} 不同，已按检查点配置构建。")
        if width != WIDTH_MULTIPLIER:
            print(f"警告: 检查点宽度为 {width}，与当前设置 {WIDTH_MULTIPLIER} 不同，已按检查点配置构建。")

    model = MobileViTv2(build_opts(n_cls, width, act)).to(device)
    load_checkpoint_into_model(model, checkpoint_path)
    model.eval()
    return model, cfg


def check_class_mapping(class_names, cfg, strict=True):
    """校验数据集的类别顺序与训练时一致（ImageFolder 按目录名排序，顺序错=标签全错）。"""
    trained = cfg.get('class_names')
    if not trained:
        print("提示: 检查点里没有记录 class_names，跳过类别顺序校验。")
        return
    if list(trained) != list(class_names):
        msg = f"类别顺序/名称与训练时不一致！\n  训练时: {trained}\n  当前:   {list(class_names)}"
        if strict:
            raise RuntimeError(msg)
        print("警告: " + msg)
    else:
        print(f"类别顺序校验通过: {list(class_names)}")


def save_confusion_matrix(all_labels, all_predictions, class_names, path, title='Confusion Matrix'):
    num_classes = len(class_names)
    conf = confusion_matrix(all_labels, all_predictions, labels=list(range(num_classes)))
    fig, ax = plt.subplots(figsize=(max(6, num_classes * 0.8), max(5, num_classes * 0.7)))
    sns.heatmap(conf, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'混淆矩阵已保存到: {path}')
    return conf


def evaluate_model(model_path, dataset_path=TEST_DATA_DIR, save_results=True, result_prefix=None,
                   device=None, batch_size=BATCH_SIZE, use_amp=USE_AMP,
                   save_confusion=False, verbose=True):
    """评估单个检查点/纯权重，返回指标字典（含逐类准确率与全部预测）。"""
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(42)
    np.random.seed(42)

    # ---------- 1. 数据（预处理与训练一致）----------
    transform = build_transform(RESIZE)
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
    class_names = list(full_dataset.classes)
    num_classes = len(class_names)
    print(f'待评估数据集: {dataset_path}')
    print(f'类别数: {num_classes}, 类别名: {class_names}')

    data_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

    # ---------- 2. 模型（结构与激活以检查点自带的 config 为准）----------
    model, cfg = build_model_from_checkpoint(model_path, num_classes=num_classes, device=device)
    check_class_mapping(class_names, cfg)

    # ---------- 3. 推理 ----------
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = [0] * num_classes
    class_total = [0] * num_classes
    all_predictions = []
    all_labels = []
    autocast_enabled = bool(use_amp) and device.type == 'cuda'

    with torch.no_grad():
        iterator = tqdm(data_loader, desc=f'评估 {os.path.basename(model_path)}', disable=not verbose)
        for images, targets in iterator:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=autocast_enabled):
                outputs = model(images)
                loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            pred_list = predicted.cpu().numpy().tolist()
            label_list = targets.cpu().numpy().tolist()
            all_predictions.extend(pred_list)
            all_labels.extend(label_list)
            for label, pred in zip(label_list, pred_list):
                class_total[label] += 1
                if label == pred:
                    class_correct[label] += 1
            if verbose:
                iterator.set_postfix(loss=loss.item(), acc=100. * correct / total)

    if total == 0:
        raise RuntimeError(f'{dataset_path} 中没有可用样本')

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    per_class_acc = [100.0 * class_correct[i] / class_total[i] if class_total[i] > 0 else None
                     for i in range(num_classes)]
    valid_accs = [a for a in per_class_acc if a is not None]
    macro_avg = float(np.mean(valid_accs)) if valid_accs else 0.0

    print(f'评估完成: 准确率 = {avg_acc:.2f}%, 平均类别准确率 = {macro_avg:.2f}%, 损失 = {avg_loss:.4f}')

    # ---------- 4. 落盘 ----------
    if save_results and result_prefix:
        os.makedirs(RESULT_DIR, exist_ok=True)
        result_file = os.path.join(RESULT_DIR, f'{result_prefix}_results.txt')
        with open(result_file, 'w', encoding='utf-8') as f:
            f.write(f'模型: {os.path.basename(model_path)}\n')
            f.write(f'数据集: {dataset_path}\n')
            f.write(f'类别数: {num_classes}\n类别名: {class_names}\n')
            f.write(f'检查点配置: {cfg}\n')
            f.write(f'总体准确率: {avg_acc:.2f}%\n')
            f.write(f'平均类别准确率: {macro_avg:.2f}%\n')
            f.write(f'损失: {avg_loss:.4f}\n\n各类别准确率:\n')
            for i, name in enumerate(class_names):
                if per_class_acc[i] is None:
                    f.write(f'  {name}: 无样本\n')
                else:
                    f.write(f'  {name}: {per_class_acc[i]:.2f}% ({class_correct[i]}/{class_total[i]})\n')
            f.write('\n分类报告:\n')
            f.write(classification_report(all_labels, all_predictions,
                                          labels=list(range(num_classes)), target_names=class_names,
                                          digits=4, zero_division=0))
        print(f'结果已保存到: {result_file}')

        if save_confusion:
            save_confusion_matrix(all_labels, all_predictions, class_names,
                                  os.path.join(RESULT_DIR, f'{result_prefix}_confusion_matrix.png'),
                                  title=f'Confusion Matrix - {os.path.basename(model_path)}')

    return {
        'model_name': os.path.basename(model_path),
        'model_path': model_path,
        'overall_accuracy': avg_acc,
        'macro_accuracy': macro_avg,
        'loss': avg_loss,
        'class_accuracies': per_class_acc,
        'class_names': class_names,
        'class_correct': class_correct,
        'class_total': class_total,
        'total_correct': correct,
        'total_samples': total,
        'predictions': all_predictions,
        'labels': all_labels,
        'config': cfg,
    }


print('公共函数已就绪: build_opts / build_transform / build_model_from_checkpoint / evaluate_model')


公共函数已就绪: build_opts / build_transform / build_model_from_checkpoint / evaluate_model


### 批量评估：把微调过程中的检查点逐个评一遍

需要先运行第 1 个代码单元（公共函数）。


In [4]:
# ================== 批量评估配置 ==================
PTH_PATTERN = './checkpoint_epoch_*.pth'                        # 训练脚本每轮保存的检查点
BEST_MODEL = 'x.pth'                  # 与训练脚本的 SAVE_PATH 保持一致
SAVE_CONFUSION_MATRIX = False                                   # 批量评估时是否保存混淆矩阵
SAVE_INDIVIDUAL_RESULTS = True                                  # 每个模型单独保存一份结果

if __name__ == '__main__':
    print('=' * 60)
    print('批量评估（MobileViTv2 0.75x / 10 类 / relu）')
    print('=' * 60)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'使用设备: {device}')

    pth_files = sorted(glob.glob(PTH_PATTERN))
    if not pth_files and BEST_MODEL and os.path.isfile(BEST_MODEL):
        print(f'没有匹配到 {PTH_PATTERN}，改用最佳模型 {BEST_MODEL}')
        pth_files = [BEST_MODEL]

    if not pth_files:
        print(f'错误: 没有找到可用模型（模式 {PTH_PATTERN}；最佳模型 {BEST_MODEL}）。请先运行训练脚本。')
    else:
        print(f'找到 {len(pth_files)} 个模型文件:')
        for i, pth_file in enumerate(pth_files, 1):
            print(f'  {i:2d}. {pth_file}')

        print('\n开始批量评估...')
        all_results = []
        for pth_file in pth_files:
            model_name = os.path.basename(pth_file)
            result_prefix = os.path.splitext(model_name)[0]
            try:
                result = evaluate_model(
                    model_path=pth_file,
                    dataset_path=TEST_DATA_DIR,
                    save_results=SAVE_INDIVIDUAL_RESULTS,
                    result_prefix=result_prefix,
                    device=device,
                    save_confusion=SAVE_CONFUSION_MATRIX,
                )
                all_results.append(result)
                print(f"[OK] 完成评估: {model_name} - 准确率 {result['overall_accuracy']:.2f}%")
            except Exception as exc:
                # 类别数不匹配、分类头对不上等问题都会走到这里，直接跳过并说明原因
                print(f"[SKIP] 跳过 {model_name}: {type(exc).__name__}: {exc}")
            print('-' * 40)

        if not all_results:
            print('错误: 没有成功评估任何模型。')
        else:
            all_results.sort(key=lambda x: x['overall_accuracy'], reverse=True)

            print('\n' + '=' * 60)
            print('所有模型评估结果（按准确率降序）:')
            print('=' * 60)
            for i, result in enumerate(all_results, 1):
                print(f"{i:2d}. {result['model_name']:28s} | 准确率 {result['overall_accuracy']:6.2f}% | "
                      f"平均类别 {result['macro_accuracy']:6.2f}% | 损失 {result['loss']:.4f}")

            print('\n准确率前 3 的模型:')
            for i, result in enumerate(all_results[:3], 1):
                print(f"第{i}名: {result['model_name']}")
                print(f"  - 准确率: {result['overall_accuracy']:.2f}%")
                print(f"  - 平均类别准确率: {result['macro_accuracy']:.2f}%")
                print(f"  - 损失: {result['loss']:.4f}")
                print(f"  - 正确样本数: {result['total_correct']}/{result['total_samples']}")

            if SAVE_INDIVIDUAL_RESULTS:
                os.makedirs(RESULT_DIR, exist_ok=True)
                summary_file = os.path.join(RESULT_DIR, 'summary_top3.txt')
                with open(summary_file, 'w', encoding='utf-8') as f:
                    f.write('模型评估汇总 - 准确率排序\n')
                    f.write('=' * 50 + '\n\n')
                    f.write('准确率前 3 的模型:\n')
                    f.write('-' * 50 + '\n')
                    for i, result in enumerate(all_results[:3], 1):
                        f.write(f"\n第{i}名: {result['model_name']}\n")
                        f.write(f"  总体准确率: {result['overall_accuracy']:.2f}%\n")
                        f.write(f"  平均类别准确率: {result['macro_accuracy']:.2f}%\n")
                        f.write(f"  损失: {result['loss']:.4f}\n")
                        f.write(f"  正确样本数: {result['total_correct']}/{result['total_samples']}\n")
                    f.write('\n所有模型排名:\n')
                    f.write('-' * 50 + '\n')
                    for i, result in enumerate(all_results, 1):
                        f.write(f"{i:3d}. {result['model_name']:28s} | 准确率 {result['overall_accuracy']:6.2f}% | "
                                f"平均类别 {result['macro_accuracy']:6.2f}%\n")
                print(f'\n汇总结果已保存到: {summary_file}')
        print('=' * 60)


批量评估（MobileViTv2 0.75x / 10 类 / relu）
使用设备: cuda
找到 7 个模型文件:
   1. .\checkpoint_epoch_1.pth
   2. .\checkpoint_epoch_2.pth
   3. .\checkpoint_epoch_3.pth
   4. .\checkpoint_epoch_4.pth
   5. .\checkpoint_epoch_5.pth
   6. .\checkpoint_epoch_6.pth
   7. .\checkpoint_epoch_7.pth

开始批量评估...
待评估数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
检查点配置: 类别数=10 宽度=0.75 激活=relu 类别名=['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
  从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）
  已加载 269/269 个张量
类别顺序校验通过: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


评估 checkpoint_epoch_1.pth: 100%|██████████| 19/19 [00:01<00:00, 13.14it/s, acc=16, loss=2.26]  


评估完成: 准确率 = 16.00%, 平均类别准确率 = 15.52%, 损失 = 2.2624
结果已保存到: model_evaluation_results\checkpoint_epoch_1_results.txt
[OK] 完成评估: checkpoint_epoch_1.pth - 准确率 16.00%
----------------------------------------
待评估数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
检查点配置: 类别数=10 宽度=0.75 激活=relu 类别名=['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
  从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）
  已加载 269/269 个张量
类别顺序校验通过: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


评估 checkpoint_epoch_2.pth: 100%|██████████| 19/19 [00:01<00:00, 16.50it/s, acc=73, loss=0.735]  


评估完成: 准确率 = 73.00%, 平均类别准确率 = 73.05%, 损失 = 0.8872
结果已保存到: model_evaluation_results\checkpoint_epoch_2_results.txt
[OK] 完成评估: checkpoint_epoch_2.pth - 准确率 73.00%
----------------------------------------
待评估数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
检查点配置: 类别数=10 宽度=0.75 激活=relu 类别名=['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
  从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）
  已加载 269/269 个张量
类别顺序校验通过: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


...

评估完成: 准确率 = 88.33%, 平均类别准确率 = 88.50%, 损失 = 0.4210
结果已保存到: model_evaluation_results\checkpoint_epoch_5_results.txt
[OK] 完成评估: checkpoint_epoch_5.pth - 准确率 88.33%
----------------------------------------
待评估数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
检查点配置: 类别数=10 宽度=0.75 激活=relu 类别名=['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
  从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）
  已加载 269/269 个张量
类别顺序校验通过: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


评估 checkpoint_epoch_6.pth: 100%|██████████| 19/19 [00:01<00:00, 16.49it/s, acc=87, loss=0.677]  


评估完成: 准确率 = 87.00%, 平均类别准确率 = 86.85%, 损失 = 0.4017
结果已保存到: model_evaluation_results\checkpoint_epoch_6_results.txt
[OK] 完成评估: checkpoint_epoch_6.pth - 准确率 87.00%
----------------------------------------
待评估数据集: C:\transformerFPGA\MobileViT\Dataset\Driver\test1_p
类别数: 10, 类别名: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
检查点配置: 类别数=10 宽度=0.75 激活=relu 类别名=['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
  从键 'ema_model_state_dict' 中提取模型权重（EMA 权重，推理用）
  已加载 269/269 个张量
类别顺序校验通过: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


评估 checkpoint_epoch_7.pth: 100%|██████████| 19/19 [00:01<00:00, 16.30it/s, acc=85.3, loss=1.18] 

评估完成: 准确率 = 85.33%, 平均类别准确率 = 85.32%, 损失 = 0.4439
结果已保存到: model_evaluation_results\checkpoint_epoch_7_results.txt
[OK] 完成评估: checkpoint_epoch_7.pth - 准确率 85.33%
----------------------------------------

所有模型评估结果（按准确率降序）:
 1. checkpoint_epoch_5.pth       | 准确率  88.33% | 平均类别  88.50% | 损失 0.4210
 2. checkpoint_epoch_6.pth       | 准确率  87.00% | 平均类别  86.85% | 损失 0.4017
 3. checkpoint_epoch_4.pth       | 准确率  86.00% | 平均类别  86.04% | 损失 0.4408
 4. checkpoint_epoch_7.pth       | 准确率  85.33% | 平均类别  85.32% | 损失 0.4439
 5. checkpoint_epoch_3.pth       | 准确率  84.33% | 平均类别  84.35% | 损失 0.4680
 6. checkpoint_epoch_2.pth       | 准确率  73.00% | 平均类别  73.05% | 损失 0.8872
 7. checkpoint_epoch_1.pth       | 准确率  16.00% | 平均类别  15.52% | 损失 2.2624

准确率前 3 的模型:
第1名: checkpoint_epoch_5.pth
  - 准确率: 88.33%
  - 平均类别准确率: 88.50%
  - 损失: 0.4210
  - 正确样本数: 265/300
第2名: checkpoint_epoch_6.pth
  - 准确率: 87.00%
  - 平均类别准确率: 86.85%
  - 损失: 0.4017
  - 正确样本数: 261/300
第3名: checkpoint_epoch_4.pth
  - 准确率: 86.00%
  - 平均类别

In [ ]:
# ================== 单模型详细评估（逐类准确率 / 分类报告 / 混淆矩阵） ==================
MODEL_PATH = 'mobilevitv2_0.75_10cls_relu.pth'   # 也可以换成某个 checkpoint_epoch_N.pth 或纯权重文件
SAVE_CONFUSION_MATRIX = True
CONFUSION_MATRIX_PATH = 'confusion_matrix.png'
SAVE_REPORT = True
REPORT_PATH = 'evaluation_results.txt'

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'使用设备: {device}')

    result = evaluate_model(MODEL_PATH, TEST_DATA_DIR, save_results=False, device=device)
    class_names = result['class_names']
    num_classes = len(class_names)

    print(f'\n{"=" * 30} 各类别准确率 {"=" * 30}')
    for i, name in enumerate(class_names):
        if result['class_total'][i] == 0:
            print(f'{name}({i}): 无样本')
        else:
            print(f"{name}({i}): {result['class_accuracies'][i]:.2f}% "
                  f"({result['class_correct'][i]}/{result['class_total'][i]})")

    print(f'\n{"=" * 30} 详细分类报告 {"=" * 30}')
    report = classification_report(result['labels'], result['predictions'],
                                   labels=list(range(num_classes)), target_names=class_names,
                                   digits=4, zero_division=0)
    print(report)
    print(f"平均类别准确率 (Macro Average): {result['macro_accuracy']:.2f}%")

    if SAVE_CONFUSION_MATRIX:
        save_confusion_matrix(result['labels'], result['predictions'], class_names,
                              CONFUSION_MATRIX_PATH,
                              title=f"Confusion Matrix - {result['model_name']}")
        print(f'\n{"=" * 30} 各类别召回率 {"=" * 30}')
        for i, name in enumerate(class_names):
            if result['class_total'][i] > 0:
                recall = 100.0 * result['class_correct'][i] / result['class_total'][i]
                print(f'{name}({i}): {recall:.2f}%')

    if SAVE_REPORT:
        with open(REPORT_PATH, 'w', encoding='utf-8') as f:
            f.write('评估结果 - MobileViTv2 0.75x / 10 类 / relu\n')
            f.write('=' * 50 + '\n')
            f.write(f'数据集: {TEST_DATA_DIR}\n')
            f.write(f'模型: {MODEL_PATH}\n')
            f.write(f"检查点配置: {result['config']}\n")
            f.write(f"总体准确率: {result['overall_accuracy']:.2f}%\n")
            f.write(f"平均类别准确率: {result['macro_accuracy']:.2f}%\n")
            f.write(f"损失: {result['loss']:.4f}\n\n各类别准确率:\n")
            for i, name in enumerate(class_names):
                if result['class_total'][i] == 0:
                    f.write(f'{name}({i}): 无样本\n')
                else:
                    f.write(f"{name}({i}): {result['class_accuracies'][i]:.2f}% "
                            f"({result['class_correct'][i]}/{result['class_total'][i]})\n")
            f.write('\n分类报告:\n')
            f.write(report)
        print(f'\n详细结果已保存到: {REPORT_PATH}')


### 可选：ONNX 导出与 ONNX Runtime 评估

只需要 PyTorch 精度评估的话，可以跳过这两节。

**注意**：本机 torch 2.9.1 的 `torch.onnx.export` 默认 `dynamo=True`。实测该导出器在此环境下会打印 emoji
并在 GBK 控制台抛出 `UnicodeEncodeError`，而且不接受 `dynamic_axes`；因此下面的导出显式使用
`dynamo=False`（TorchScript 导出器），实测可动态 batch、与 PyTorch 输出最大误差 1.3e-06。


In [ ]:
# ================== ONNX 导出（10 类 / 0.75x / relu） ==================
PTH_PATH = 'mobilevitv2_0.75_10cls_relu.pth'      # 训练产出的检查点或纯权重
ONNX_PATH = 'MobileViTv2-0.75-10cls-relu.onnx'
INPUT_SIZE = (1, 3, 256, 256)                     # 数据是 256x256，能被下采样倍数 32 整除
DYNAMIC_BATCH = True                              # 只把 batch 维设为动态
OPSET = 17
# torch>=2.9 的 torch.onnx.export 默认 dynamo=True，但该导出器在本机（GBK 控制台）会因打印 emoji
# 抛 UnicodeEncodeError，且不接受 dynamic_axes，所以这里显式使用 TorchScript 导出器。
DYNAMO = False


def export_onnx(model, onnx_path, input_size=INPUT_SIZE, dynamic_batch=DYNAMIC_BATCH,
                opset=OPSET, dynamo=DYNAMO):
    device = next(model.parameters()).device
    dummy_input = torch.randn(*input_size, device=device)
    dynamic_axes = None
    if dynamic_batch:
        dynamic_axes = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}

    kwargs = dict(export_params=True, opset_version=opset, do_constant_folding=True,
                  input_names=['input'], output_names=['output'], dynamic_axes=dynamic_axes)
    try:
        torch.onnx.export(model, dummy_input, onnx_path, dynamo=dynamo, **kwargs)
    except TypeError:
        # 老版本 torch 的 torch.onnx.export 没有 dynamo 形参
        torch.onnx.export(model, dummy_input, onnx_path, **kwargs)
    return onnx_path


if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_classes = len(datasets.ImageFolder(root=TEST_DATA_DIR).classes)
    print(f'数据集类别数: {num_classes}（必须与训练时一致）')

    model, cfg = build_model_from_checkpoint(PTH_PATH, num_classes=num_classes, device=device)
    onnx_path = export_onnx(model, ONNX_PATH)
    print(f'ONNX 模型已导出至: {onnx_path}')

    # ---------- 导出后自检 ----------
    import onnx
    m = onnx.load(onnx_path)
    onnx.checker.check_model(m)
    print('onnx.checker 校验通过')

    def _shape(t):
        return [d.dim_param if d.dim_param else d.dim_value for d in t.type.tensor_type.shape.dim]

    print('  输入:', [(i.name, _shape(i)) for i in m.graph.input])
    print('  输出:', [(o.name, _shape(o)) for o in m.graph.output])

    try:
        import onnxruntime as ort
        providers = ['CPUExecutionProvider']
        sess = ort.InferenceSession(onnx_path, providers=providers)
        x = torch.randn(2, 3, 256, 256)
        with torch.no_grad():
            y_pt = model(x.to(device)).float().cpu().numpy()
        y_ort = sess.run(None, {'input': x.numpy().astype(np.float32)})[0]
        print(f'  ONNX Runtime 输出: {y_ort.shape}, 与 PyTorch 最大绝对误差: '
              f'{np.abs(y_ort - y_pt).max():.3e}')
    except ImportError:
        print('  未安装 onnxruntime，跳过数值一致性检查')


In [ ]:
# ================== 用 ONNX Runtime 评估导出的 ONNX 模型 ==================
ONNX_MODEL_PATH = 'MobileViTv2-0.75-10cls-relu.onnx'
SAVE_ONNX_RESULTS = True


def evaluate_onnx_model(onnx_path=ONNX_MODEL_PATH, dataset_path=TEST_DATA_DIR,
                        batch_size=BATCH_SIZE, save_results=SAVE_ONNX_RESULTS):
    """用 ONNX Runtime 跑一遍测试集（预处理与训练/ PyTorch 评估完全一致）。"""
    import onnxruntime as ort

    providers = (['CUDAExecutionProvider', 'CPUExecutionProvider']
                 if 'CUDAExecutionProvider' in ort.get_available_providers()
                 else ['CPUExecutionProvider'])
    print(f'可用提供程序: {ort.get_available_providers()}')
    print(f'使用提供程序: {providers}')

    try:
        session = ort.InferenceSession(onnx_path, providers=providers)
    except Exception as exc:
        print(f'[FAIL] ONNX 模型加载失败: {type(exc).__name__}: {exc}')
        return None
    print(f'[OK] ONNX 模型加载成功: {onnx_path}')

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    print(f'输入节点: {input_name} {session.get_inputs()[0].shape}')
    print(f'输出节点: {output_name} {session.get_outputs()[0].shape}')

    transform = build_transform(RESIZE)
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
    class_names = list(full_dataset.classes)
    num_classes = len(class_names)
    print(f'待评估数据集: {dataset_path}')
    print(f'类别数: {num_classes}, 类别名: {class_names}')

    data_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS)

    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct = 0
    total = 0
    class_correct = [0] * num_classes
    class_total = [0] * num_classes
    all_predictions = []
    all_labels = []

    for images, targets in tqdm(data_loader, desc='ONNX 评估'):
        inputs = images.numpy().astype(np.float32)
        outputs = session.run([output_name], {input_name: inputs})[0]
        outputs_t = torch.from_numpy(outputs)
        targets_t = torch.from_numpy(targets.numpy())

        loss = criterion(outputs_t, targets_t)
        running_loss += loss.item() * len(images)

        _, predicted = outputs_t.max(1)
        correct += (predicted == targets_t).sum().item()
        total += targets_t.size(0)

        pred_list = predicted.numpy().tolist()
        label_list = targets_t.numpy().tolist()
        all_predictions.extend(pred_list)
        all_labels.extend(label_list)
        for label, pred in zip(label_list, pred_list):
            class_total[label] += 1
            if label == pred:
                class_correct[label] += 1

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    per_class = [100.0 * class_correct[i] / class_total[i] if class_total[i] > 0 else None
                 for i in range(num_classes)]
    valid = [a for a in per_class if a is not None]
    macro_avg = float(np.mean(valid)) if valid else 0.0

    print(f'ONNX 评估完成: 准确率 = {avg_acc:.2f}%, 平均类别准确率 = {macro_avg:.2f}%, 损失 = {avg_loss:.4f}')

    if save_results:
        os.makedirs(RESULT_DIR, exist_ok=True)
        result_file = os.path.join(RESULT_DIR, 'onnx_eval_results.txt')
        with open(result_file, 'w', encoding='utf-8') as f:
            f.write(f'模型: {os.path.basename(onnx_path)}\n')
            f.write(f'数据集: {dataset_path}\n')
            f.write(f'总体准确率: {avg_acc:.2f}%\n')
            f.write(f'平均类别准确率: {macro_avg:.2f}%\n')
            f.write(f'损失: {avg_loss:.4f}\n\n各类别准确率:\n')
            for i, name in enumerate(class_names):
                if per_class[i] is None:
                    f.write(f'  {name}: 无样本\n')
                else:
                    f.write(f'  {name}: {per_class[i]:.2f}% ({class_correct[i]}/{class_total[i]})\n')
            f.write('\n分类报告:\n')
            f.write(classification_report(all_labels, all_predictions,
                                          labels=list(range(num_classes)), target_names=class_names,
                                          digits=4, zero_division=0))
        print(f'结果已保存到: {result_file}')

    return {
        'model_name': os.path.basename(onnx_path),
        'overall_accuracy': avg_acc,
        'macro_accuracy': macro_avg,
        'loss': avg_loss,
        'class_accuracies': per_class,
        'total_correct': correct,
        'total_samples': total,
        'predictions': all_predictions,
        'labels': all_labels,
        'class_names': class_names,
    }


if __name__ == '__main__':
    if not os.path.isfile(ONNX_MODEL_PATH):
        print(f'找不到 {ONNX_MODEL_PATH}，请先运行上面的 ONNX 导出单元。')
    else:
        result = evaluate_onnx_model()
        if result:
            print(f"\n最终结果: 准确率 = {result['overall_accuracy']:.2f}%, "
                  f"宏平均准确率 = {result['macro_accuracy']:.2f}%")


### 可选：权重工具（提取纯权重 / 多个检查点融合）

给 FPGA 侧部署或模型集成时用，平时可以跳过。


In [ ]:
# ================== 从完整检查点里提取纯权重 ==================
CHECKPOINT_PATH = 'mobilevitv2_0.75_10cls_relu.pth'
OUTPUT_PATH = 'mobilevitv2_0.75_10cls_relu_weights.pth'

if __name__ == '__main__':
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    state = extract_state_dict(ckpt)          # 优先取 model_state_dict（训练脚本里它就是最佳权重）
    torch.save(state, OUTPUT_PATH)

    cfg = ckpt.get('config', {}) if isinstance(ckpt, dict) else {}
    print(f'已保存纯权重: {OUTPUT_PATH}')
    print(f'  张量数: {len(state)}，参数量: {sum(v.numel() for v in state.values()):,}')
    print(f"  配置: {cfg.get('num_classes')} 类, 宽度 {cfg.get('width_multiplier')}, 激活 {cfg.get('activation')}")
    if isinstance(ckpt, dict):
        print(f"  最佳验证准确率: {ckpt.get('best_val_acc')}, 权重来源: {ckpt.get('best_weights_source')}")


In [ ]:
# ================== 权重融合（多个检查点加权平均，可选工具） ==================
from collections import OrderedDict

WEIGHTS = [0.6, 0.4]                            # 与 CHECKPOINT_PATHS 一一对应，和必须为 1
CHECKPOINT_PATHS = [
    'checkpoint_epoch_30.pth',                  # 基准模型：分类头以它为准
    'checkpoint_epoch_28.pth',
]
OUTPUT_PATH = 'fused_0.75_10cls_relu.pth'
KEEP_BASE_CLASSIFIER = False                    # True = 分类头完全采用基准模型，不参与平均
DEVICE = 'cpu'


def average_weights(paths, weights, output_path, device='cpu', keep_base_classifier=False):
    """按权重融合多个检查点；形状冲突的分类头保留基准模型的值。"""
    assert len(paths) == len(weights), '路径与权重数量必须一致'
    assert abs(sum(weights) - 1.0) < 1e-6, f'权重和={sum(weights)}，必须等于 1'

    state_dicts = []
    for path in paths:
        print(f'加载: {path}')
        ckpt = torch.load(path, map_location=device, weights_only=False)
        state_dicts.append(extract_state_dict(ckpt))

    sd0 = state_dicts[0]
    for path, sd in zip(paths[1:], state_dicts[1:]):
        if set(sd.keys()) != set(sd0.keys()):
            diff = sorted(set(sd.keys()) ^ set(sd0.keys()))
            raise RuntimeError(f'键不匹配（{path}）: {diff[:10]}')

    # ---------- 预扫描：找出形状冲突的键 ----------
    special_keys = set()
    for k in sd0:
        shapes = {tuple(sd[k].shape) for sd in state_dicts}
        if len(shapes) > 1:
            if 'classifier' in k:
                special_keys.add(k)
                print(f'  [冲突处理] {k}: 形状 {shapes}，保留 {paths[0]} 的值')
            else:
                raise RuntimeError(f'{k} 形状冲突且不是分类头: {shapes}')

    if keep_base_classifier:
        special_keys.update(k for k in sd0 if k.startswith('classifier.'))
        print(f'  [配置] 分类头不参与平均，共 {len(special_keys)} 个键保留基准模型的值')

    # ---------- 初始化累加器 ----------
    fused = OrderedDict()
    for k, v in sd0.items():
        if k in special_keys or not torch.is_floating_point(v):
            fused[k] = v.detach().clone().to(device=device)      # 整数 buffer 只保留基准值
        else:
            fused[k] = torch.zeros_like(v, dtype=torch.float64, device=device)

    # ---------- 逐文件加权累加 ----------
    for path, w, sd in zip(paths, weights, state_dicts):
        print(f'融合: {path} (w={w})')
        for k in fused:
            if k in special_keys or not torch.is_floating_point(fused[k]):
                continue
            fused[k].add_(sd[k].to(dtype=torch.float64, device=device) * w)

    # ---------- 恢复原始 dtype ----------
    for k, v in fused.items():
        if torch.is_floating_point(v):
            fused[k] = v.to(sd0[k].dtype)

    torch.save(fused, output_path)
    print(f'已保存融合权重: {output_path}（{len(fused)} 个张量）')
    return fused


if __name__ == '__main__':
    missing = [p for p in CHECKPOINT_PATHS if not os.path.isfile(p)]
    if missing:
        print(f'找不到这些文件，请先修改 CHECKPOINT_PATHS: {missing}')
    else:
        average_weights(CHECKPOINT_PATHS, WEIGHTS, OUTPUT_PATH, DEVICE, KEEP_BASE_CLASSIFIER)
        print('提示: 融合结果是纯 state_dict，可直接填到前面评估单元的 MODEL_PATH 里使用。')
